In [1]:
import numpy as np
from tensorflow.keras.layers import Embedding, Flatten, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences

docs = [ 'additional income',
         'best price',
         'big bucks',
         'cash bonus',
         'earn extra cash',
         'spring savings certificate',
         'valero gas marketing',
         'all domestic employees',
         'nominations for oct',
         'confirmation from spinner']

labels = np.array([1,1,1,1,1,0,0,0,0,0])

vocab_size = 50
encoded_docs = [one_hot(d, vocab_size) for d in docs]
print(encoded_docs)

max_length = 4
padded_docs = pad_sequences(encoded_docs, maxlen=max_length, padding='post')
print(padded_docs)

model = Sequential()
model.add(Embedding(vocab_size, 8, input_length=max_length))
model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.fit(padded_docs, labels, epochs=50, verbose=0)

loss, accuracy = model.evaluate(padded_docs, labels, verbose=0)
print('정확도=', accuracy)

test_doc = ['big income']
encoded_docs = [one_hot(d, vocab_size) for d in test_doc]
padded_docs = pad_sequences(encoded_docs, maxlen=max_length, padding='post')

print(model.predict(padded_docs))

[[19, 8], [8, 16], [30, 28], [8, 20], [19, 11, 8], [23, 19, 24], [4, 15, 17], [21, 45, 17], [35, 22, 36], [41, 13, 44]]
[[19  8  0  0]
 [ 8 16  0  0]
 [30 28  0  0]
 [ 8 20  0  0]
 [19 11  8  0]
 [23 19 24  0]
 [ 4 15 17  0]
 [21 45 17  0]
 [35 22 36  0]
 [41 13 44  0]]


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


정확도= 1.0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
[[0.5684202]]


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense, Dropout
import re

print("데이터를 다운로드하는 중입니다...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)
print(f"훈련 데이터: {len(x_train)}개, 테스트 데이터: {len(x_test)}개\n")

word_to_index = imdb.get_word_index()
word_to_index = {k: (v + 3) for k, v in word_to_index.items()}
word_to_index["<PAD>"] = 0
word_to_index["<START>"] = 1
word_to_index["<UNK>"] = 2
word_to_index["<UNUSED>"] = 3

index_to_word = {value: key for key, value in word_to_index.items()}

max_length = 100
x_train = pad_sequences(x_train, maxlen=max_length)
x_test = pad_sequences(x_test, maxlen=max_length)

vocab_size = 10000

model = Sequential()
model.add(Embedding(vocab_size, 64, input_length=max_length))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

print("=== 모델 구조 ===")
model.summary()

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

print("\n모델 훈련을 시작합니다...")
history = model.fit(x_train, y_train,
                    batch_size=64, epochs=10,
                    verbose=1,
                    validation_data=(x_test, y_test))

print("\n=== 모델 평가 결과 ===")
results = model.evaluate(x_test, y_test, verbose=2)
print(f"손실(Loss): {results[0]:.4f} / 정확도(Accuracy): {results[1]:.4f}")

review = "What can I say about this movie that was already said? It is my favorite time travel sci-fi, adventure epic comedy in the 80's and I love this movie to death! When I saw this movie I was thrown out by its theme. An excellent sci-fi, adventure epic, I LOVE the 80s. It's simple the greatest time travel movie ever happened in the history of world cinema. I love this movie to death, I love, LOVE, love it!"

review = re.sub("[^0-9a-zA-Z ]", "", review).lower()

review_encoding = []
for w in review.split():
    index = word_to_index.get(w, 2)
    if index < 10000:
        review_encoding.append(index)
    else:
        review_encoding.append(2)

test_input = pad_sequences([review_encoding], maxlen=max_length)

value = model.predict(test_input)
prediction_score = value[0][0]

print("\n=== 새로운 리뷰 예측 결과 ===")
print(f"입력한 리뷰의 긍정 확률: {prediction_score * 100:.2f}%")
if prediction_score > 0.5:
    print("👉 긍정적인 리뷰입니다.")
else:
    print("👉 부정적인 리뷰입니다.")

데이터를 다운로드하는 중입니다...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
훈련 데이터: 25000개, 테스트 데이터: 25000개

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
=== 모델 구조 ===


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


모델 훈련을 시작합니다...
Epoch 1/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - accuracy: 0.7644 - loss: 0.4618 - val_accuracy: 0.8479 - val_loss: 0.3416
Epoch 2/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.9360 - loss: 0.1771 - val_accuracy: 0.8286 - val_loss: 0.4165
Epoch 3/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 20ms/step - accuracy: 0.9919 - loss: 0.0328 - val_accuracy: 0.8348 - val_loss: 0.5322
Epoch 4/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 0.9988 - loss: 0.0074 - val_accuracy: 0.8376 - val_loss: 0.6040
Epoch 5/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 0.9998 - loss: 0.0025 - val_accuracy: 0.8380 - val_loss: 0.6639
Epoch 6/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 1.0000 - loss: 0.0011 - val_accuracy: 0.8396 - val_loss: 0.7020
Epoch 7/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - accuracy: 1.0000 - loss: 6.1966e-04 - val_accuracy: 0.8410 - val_loss: 0.7360
Epoch 8/10
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 1.0000 -